# Register MCP Policy

Creates the service policy function and supporting table for the 4-persona
MCP Services (architect, reviewer, worker, dispatcher).

## Architecture

1. **persona_assignments** table — maps persona → gated tools (ASK-gate)
2. **persona_tool_policy** function — SQL UDF (`event VARIANT → VARIANT`)
   that returns ASK/ALLOW/DENY based on tool name and actor context
3. **Policy attachment** — links the function to each MCP Service at ON_CALL

## Beta Limitation (as of 2026-08)

> **Service policy attachment is UI-only during Beta.**
> There is no public API endpoint for attaching policies to MCP Services.
> The `PUT /service-policies` endpoint used previously does not exist.
>
> **Manual step required:** After this notebook runs, attach the policy
> via Catalog Explorer → each service → Policies tab → attach
> `persona_tool_policy` as ON_CALL policy.
>
> **Prerequisite:** An account admin must enable "Unity AI Gateway beta
> features" from the account console Previews page.
>
> See: https://docs.databricks.com/aws/en/data-governance/unity-catalog/service-policies/create-service-policy/

## Parameters

| Parameter | Description |
|-----------|-------------|
| `catalog` | UC catalog |
| `schema` | UC schema |
| `architect_mcp_service_name` | Architect service name |
| `reviewer_mcp_service_name` | Reviewer service name |
| `worker_mcp_service_name` | Worker service name |
| `dispatcher_mcp_service_name` | Dispatcher service name |
| `workers_group_name` | Group receiving EXECUTE on policy function |

In [0]:
%pip install --upgrade databricks-sdk
dbutils.library.restartPython()

In [0]:
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
architect_mcp_service_name = dbutils.widgets.get("architect_mcp_service_name")
reviewer_mcp_service_name = dbutils.widgets.get("reviewer_mcp_service_name")
worker_mcp_service_name = dbutils.widgets.get("worker_mcp_service_name")
dispatcher_mcp_service_name = dbutils.widgets.get("dispatcher_mcp_service_name")

import os

# Derive path to policies/ directory from this notebook's location
nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
nb_dir = os.path.dirname(nb_path)
policies_dir = os.path.join("/Workspace" + nb_dir, "policies")

# Read and execute the DDL for persona_assignments table
ddl_file = os.path.join(policies_dir, "persona_assignments_ddl.sql")
with open(ddl_file, "r") as f:
    ddl = f.read()

ddl = ddl.replace("{{catalog}}", catalog).replace("{{schema}}", schema)

# Filter out comment-only lines for execution
statements = [s.strip() for s in ddl.split(";") if s.strip() and not s.strip().startswith("--")]
for stmt in statements:
    if stmt.upper().startswith("CREATE"):
        spark.sql(stmt)
        print(f"✅ Table created: {catalog}.{schema}.persona_assignments")

In [0]:
# Read and execute the policy function DDL
policy_file = os.path.join(policies_dir, "persona_tool_policy.sql")
with open(policy_file, "r") as f:
    policy_ddl = f.read()

policy_ddl = policy_ddl.replace("{{catalog}}", catalog).replace("{{schema}}", schema)

# Extract the CREATE OR REPLACE FUNCTION statement
import re
match = re.search(r"(CREATE OR REPLACE FUNCTION[\s\S]+?);", policy_ddl)
if match:
    func_stmt = match.group(1)
    spark.sql(func_stmt)
    print(f"✅ Policy function created: {catalog}.{schema}.persona_tool_policy")
else:
    raise ValueError("Could not extract CREATE FUNCTION statement from policy template")

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

policy_function = f"{catalog}.{schema}.persona_tool_policy"

# Service names for reference (policy must be attached manually via UI)
service_names = [
    f"{catalog}.{schema}.{architect_mcp_service_name}",
    f"{catalog}.{schema}.{reviewer_mcp_service_name}",
    f"{catalog}.{schema}.{worker_mcp_service_name}",
    f"{catalog}.{schema}.{dispatcher_mcp_service_name}",
]

# ──────────────────────────────────────────────────────────────────────────────
# MANUAL STEP: Service policy attachment is UI-only during Beta.
#
# For each service below, go to:
#   Catalog Explorer → service → Policies tab → Attach policy
#   Function: persona_tool_policy
#   Trigger:  ON_CALL
#
# Prerequisite: Account admin must enable "Unity AI Gateway beta features"
# from the account console Previews page.
#
# Docs: https://docs.databricks.com/aws/en/data-governance/unity-catalog/service-policies/create-service-policy/
# ──────────────────────────────────────────────────────────────────────────────

print("ℹ️  Policy attachment is UI-only during Beta (no API endpoint).")
print(f"\nPolicy function: {policy_function}")
print(f"Trigger: ON_CALL")
print(f"\nAttach manually to each service via Catalog Explorer → Policies tab:")
for svc in service_names:
    print(f"  • {svc}")

# Verify policy function is callable (prerequisite for attachment)
try:
    desc = spark.sql(f"DESCRIBE FUNCTION {policy_function}").collect()
    print(f"\n✅ Policy function exists and is accessible")
except Exception as e:
    print(f"\n❌ Policy function not found: {e}")
    raise

In [0]:
workers_group_name = dbutils.widgets.get("workers_group_name")

# Grant EXECUTE on the policy function to the workers group
# (needed so the platform can evaluate it on their behalf when attached)
try:
    spark.sql(f"GRANT EXECUTE ON FUNCTION {catalog}.{schema}.persona_tool_policy TO `{workers_group_name}`")
    print(f"✅ Granted EXECUTE on policy function to `{workers_group_name}`")
except Exception as e:
    print(f"⚠️  Grant: {e}")

# Summary
print(f"\n{'='*60}")
print(f"ASK-gate policy PREPARED (manual attachment required):")
print(f"  Table:    {catalog}.{schema}.persona_assignments")
print(f"  Function: {catalog}.{schema}.persona_tool_policy")
print(f"  Trigger:  ON_CALL")
print(f"  Gates:    publish_plan, dispatch_workstream, approve_review, cancel workstream")
print(f"{'='*60}")
print(f"\nServices to attach policy to (via Catalog Explorer UI):")
for svc in service_names:
    print(f"  • {svc}")
print(f"\n⚠️  NEXT STEP: Attach policy manually via Catalog Explorer → Policies tab.")
print(f"   Prerequisite: Account admin must enable 'Unity AI Gateway beta features'.")
print(f"\nTool visibility is pre-filtered by persona-specific MCP Services.")
print(f"No DENY rules needed — callers only see tools on their service path.")